In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from datasets import load_dataset
from transformers import BertTokenizer, BertModel, ViTModel, ViTImageProcessor

import numpy as np
from PIL import Image
from tqdm import tqdm

# Data Processing

In [4]:
dataset = load_dataset("Zoe3324/flickr30k-pairs")

train_data = dataset["train"]
val_data = dataset["validation"]
test_data = dataset["test"]

for i in range(5):
    print(train_data[2*i])

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=333x500 at 0x1BBD7DD3E10>, 'caption': 'Two young guys with shaggy hair look at their hands while hanging out in the yard.', 'split': 'train', 'img_id': '0', 'filename': '1000092795.jpg'}
{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=333x500 at 0x1BD3271FD90>, 'caption': 'Two men in green shirts are standing in a yard.', 'split': 'train', 'img_id': '0', 'filename': '1000092795.jpg'}
{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=333x500 at 0x1BD32729B90>, 'caption': 'Two friends enjoy time spent together.', 'split': 'train', 'img_id': '0', 'filename': '1000092795.jpg'}
{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=500x374 at 0x1BD32729C90>, 'caption': 'Workers look down from up above on a piece of equipment.', 'split': 'train', 'img_id': '1', 'filename': '10002456.jpg'}
{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=500x374 at 0x1BD32729C90>, 'captio

In [5]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
image_processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")

# The Model

In [6]:
class CrossAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.query = nn.Linear(dim, dim)
        self.key = nn.Linear(dim, dim)
        self.value = nn.Linear(dim, dim)
        self.scale = dim ** -0.5

    def forward(self, q, k, v):
        Q = self.query(q)
        K = self.key(k)
        V = self.value(v)

        attn = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        attn = torch.softmax(attn, dim=-1)

        out = torch.matmul(attn, V)
        return out

In [7]:
class TwoStageModel(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()

        self.text_encoder = BertModel.from_pretrained("bert-base-uncased")
        self.image_encoder = ViTModel.from_pretrained("google/vit-base-patch16-224")

        dim = 768

        # Stage 1: projection heads for contrastive retrieval
        self.image_proj = nn.Linear(dim, embed_dim)
        self.text_proj = nn.Linear(dim, embed_dim)

        # Stage 2: cross-attention re-ranker
        self.cross_attn = CrossAttention(dim)
        self.match_head = nn.Sequential(
            nn.Linear(dim, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, 1)
        )

        self.temperature = nn.Parameter(torch.tensor(0.07))

    # Note: B=batch size, P=number of image patches, T=number of text tokens, D=hidden dimension(embedding size)
    def encode_image(self, pixel_values):
        """Encode images → (embedding for retrieval, token features for re-ranking)."""
        image_out = self.image_encoder(pixel_values=pixel_values).last_hidden_state  # (B, P, D)
        image_emb = F.normalize(self.image_proj(image_out[:, 0]), dim=-1)  # CLS token → (B, embed_dim)
        return image_emb, image_out

    def encode_text(self, input_ids, attention_mask):
        """Encode text → (embedding for retrieval, token features for re-ranking)."""
        text_out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state  # (B, T, D)
        text_emb = F.normalize(self.text_proj(text_out[:, 0]), dim=-1)  # [CLS] token → (B, embed_dim)
        return text_emb, text_out

    def compute_itm(self, text_tokens, image_tokens):
        """Stage 2: cross-attention fusion → match score."""
        fused = self.cross_attn(text_tokens, image_tokens, image_tokens)  # (B, T, D)
        fused = fused.mean(dim=1)  # (B, D)
        logits = self.match_head(fused).squeeze(-1)  # (B,)
        return logits

    def forward(self, input_ids, attention_mask, pixel_values):
        """Full forward: returns (image_emb, text_emb, itm_logits)."""
        image_emb, image_tokens = self.encode_image(pixel_values)
        text_emb, text_tokens = self.encode_text(input_ids, attention_mask)
        itm_logits = self.compute_itm(text_tokens, image_tokens)
        return image_emb, text_emb, itm_logits

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = TwoStageModel().to(device)

cuda


Some weights of ViTModel were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# Trainning

In [ ]:
def collate_fn(batch):
    texts = [x["caption"] for x in batch]
    images = [x["image"].convert("RGB") for x in batch]

    # Create negative pairs by shifting captions by 1
    neg_texts = texts[1:] + texts[:1]

    all_texts = texts + neg_texts
    all_images = images + images  # each image paired with correct and wrong caption
    labels = [1.0] * len(texts) + [0.0] * len(texts)

    text_enc = tokenizer(
        all_texts,
        padding="max_length",
        truncation=True,
        max_length=32,
        return_tensors="pt"
    )

    image_enc = image_processor(
        images=all_images,
        return_tensors="pt"
    )

    return {
        "input_ids": text_enc["input_ids"],
        "attention_mask": text_enc["attention_mask"],
        "pixel_values": image_enc["pixel_values"],
        "labels": torch.tensor(labels, dtype=torch.float32),
    }

In [8]:
def train(model, batch_size=32, epochs=5, lr=3e-5):
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, collate_fn=collate_fn, num_workers=4)
    val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False, collate_fn=collate_fn, num_workers=4)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    itm_criterion = nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        # Training
        model.train()
        total_train_loss = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)

            image_emb, text_emb, itm_logits = model(input_ids, attention_mask, pixel_values)

            # Stage 1 loss: symmetric contrastive on positive pairs only (first half of batch)
            B = image_emb.size(0) // 2
            img_emb_pos = image_emb[:B]
            txt_emb_pos = text_emb[:B]
            sim = img_emb_pos @ txt_emb_pos.T / model.temperature
            contrastive_labels = torch.arange(B, device=device)
            loss_i2t = F.cross_entropy(sim, contrastive_labels)
            loss_t2i = F.cross_entropy(sim.T, contrastive_labels)
            contrastive_loss = (loss_i2t + loss_t2i) / 2

            # Stage 2 loss: ITM on all positive + negative pairs
            itm_loss = itm_criterion(itm_logits, labels)

            loss = contrastive_loss + itm_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()

        avg_train_loss = total_train_loss / len(train_loader)

        # Validation
        model.eval()
        total_val_loss = 0
        correct = 0
        total = 0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                pixel_values = batch["pixel_values"].to(device)
                labels = batch["labels"].to(device)

                image_emb, text_emb, itm_logits = model(input_ids, attention_mask, pixel_values)

                B = image_emb.size(0) // 2
                sim = image_emb[:B] @ text_emb[:B].T / model.temperature
                contrastive_labels = torch.arange(B, device=device)
                contrastive_loss = (F.cross_entropy(sim, contrastive_labels) + F.cross_entropy(sim.T, contrastive_labels)) / 2
                itm_loss = itm_criterion(itm_logits, labels)
                loss = contrastive_loss + itm_loss

                total_val_loss += loss.item()

                preds = (torch.sigmoid(itm_logits) > 0.5).float()
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        avg_val_loss = total_val_loss / len(val_loader)
        val_acc = correct / total

        print(f"Epoch {epoch+1}/{epochs} — Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Val ITM Acc: {val_acc:.4f}")

In [9]:
train(model, batch_size=32, epochs=1, lr=3e-5)

Epoch 1/1 [Val]: 100%|██████████| 159/159 [00:28<00:00,  5.54it/s]

Epoch 1/1 — Train Loss: 0.6537, Val Loss: 3.6184, Val ITM Acc: 0.6006


# Save/Load Model

In [9]:
# torch.save(model.state_dict(), 'advanced_model_weights.pt')
model.load_state_dict(torch.load('weights/Similarity+Rerank/advanced_model_weights_similarity+rerank.pt', weights_only=True))

<All keys matched successfully>

# Evaluation: 1to1

In [28]:
def eval_collate_fn(batch):
    """Collate without negatives — just return raw images and captions."""
    texts = [x["caption"] for x in batch]
    images = [x["image"].convert("RGB") for x in batch]
    return texts, images

In [29]:
def evaluate_image_to_text_1to1(model, data, num_samples=200, top_k=20):
    """Two-stage retrieval: coarse dual-encoder retrieval → cross-attention re-ranking."""
    subset = data.select(range(min(num_samples, len(data))))
    loader = DataLoader(subset, batch_size=32, shuffle=False, collate_fn=eval_collate_fn)

    all_texts = []
    all_images = []
    for texts, images in loader:
        all_texts.extend(texts)
        all_images.extend(images)

    N = len(all_texts)

    # ── Stage 1: Encode all images and texts independently ──
    print("Stage 1: Computing embeddings...")
    all_image_embs = []
    all_text_embs = []
    model.eval()
    bs = 64
    with torch.no_grad():
        # Encode all images
        for start in range(0, N, bs):
            end = min(start + bs, N)
            image_enc = image_processor(images=all_images[start:end], return_tensors="pt")
            img_emb, _ = model.encode_image(image_enc["pixel_values"].to(device))
            all_image_embs.append(img_emb.cpu())

        # Encode all texts
        for start in range(0, N, bs):
            end = min(start + bs, N)
            text_enc = tokenizer(all_texts[start:end], padding="max_length", truncation=True, max_length=32, return_tensors="pt")
            txt_emb, _ = model.encode_text(text_enc["input_ids"].to(device), text_enc["attention_mask"].to(device))
            all_text_embs.append(txt_emb.cpu())

    all_image_embs = torch.cat(all_image_embs)  # (N, embed_dim)
    all_text_embs = torch.cat(all_text_embs)    # (N, embed_dim)

    # Coarse similarity matrix
    coarse_sim = all_image_embs @ all_text_embs.T  # (N, N)

    # ── Stage 2: Re-rank top-k candidates with cross-attention ──
    print(f"Stage 2: Re-ranking top-{top_k} candidates...")
    score_matrix = torch.full((N, N), -1.0)  # only fill top-k entries

    with torch.no_grad():
        for i in tqdm(range(N), desc="Re-ranking"):
            # Get top-k caption candidates for image_i
            candidates = coarse_sim[i].topk(top_k).indices.tolist()

            # Prepare image_i paired with each candidate caption
            img = all_images[i]
            cand_texts = [all_texts[j] for j in candidates]
            image_enc = image_processor(images=[img] * top_k, return_tensors="pt")
            text_enc = tokenizer(cand_texts, padding="max_length", truncation=True, max_length=32, return_tensors="pt")

            itm_logits = model.compute_itm(
                model.encode_text(text_enc["input_ids"].to(device), text_enc["attention_mask"].to(device))[1],
                model.encode_image(image_enc["pixel_values"].to(device))[1],
            )
            itm_scores = torch.sigmoid(itm_logits).cpu()

            for rank, j in enumerate(candidates):
                score_matrix[i][j] = itm_scores[rank]

    def recall_at_k(score_mat, k):
        correct = 0
        for i in range(score_mat.size(0)):
            topk = score_mat[i].topk(k).indices
            if i in topk:
                correct += 1
        return correct / score_mat.size(0)

    # Image-to-text
    print()
    print(f"=== Stage 1 Only: Coarse Retrieval (N={N}) ===")
    print(f"  Recall@1:  {recall_at_k(coarse_sim, 1):.4f}")
    print(f"  Recall@5:  {recall_at_k(coarse_sim, 5):.4f}")
    print(f"  Recall@10: {recall_at_k(coarse_sim, 10):.4f}")

    print(f"\n=== Two-Stage: Coarse + Re-Rank top-{top_k} (N={N}) ===")
    print(f"  Recall@1:  {recall_at_k(score_matrix, 1):.4f}")
    print(f"  Recall@5:  {recall_at_k(score_matrix, 5):.4f}")
    print(f"  Recall@10: {recall_at_k(score_matrix, 10):.4f}")

evaluate_image_to_text_1to1(model, test_data, num_samples=200, top_k=20)

Stage 1: Computing embeddings...
Stage 2: Re-ranking top-20 candidates...


Re-ranking: 100%|██████████| 200/200 [00:29<00:00,  6.81it/s]


=== Stage 1 Only: Coarse Retrieval (N=200) ===
  Recall@1:  0.1850
  Recall@5:  0.8000
  Recall@10: 0.9600

=== Two-Stage: Coarse + Re-Rank top-20 (N=200) ===
  Recall@1:  0.1750
  Recall@5:  0.7800
  Recall@10: 0.9450


In [30]:
def evaluate_text_to_image_1to1(model, data, num_samples=200, top_k=20):
    """Two-stage TEXT-TO-IMAGE retrieval (ungrouped, 1-to-1 matching assumed)."""
    subset = data.select(range(min(num_samples, len(data))))
    loader = DataLoader(subset, batch_size=32, shuffle=False, collate_fn=eval_collate_fn)

    all_texts = []
    all_images = []
    for texts, images in loader:
        all_texts.extend(texts)
        all_images.extend(images)

    N = len(all_texts)

    # ── Stage 1: Encode all images and texts independently ──
    print("Stage 1: Computing embeddings...")
    all_image_embs = []
    all_text_embs = []
    model.eval()
    bs = 64
    with torch.no_grad():
        for start in range(0, N, bs):
            end = min(start + bs, N)
            image_enc = image_processor(images=all_images[start:end], return_tensors="pt")
            img_emb, _ = model.encode_image(image_enc["pixel_values"].to(device))
            all_image_embs.append(img_emb.cpu())

        for start in range(0, N, bs):
            end = min(start + bs, N)
            text_enc = tokenizer(all_texts[start:end], padding="max_length", truncation=True, max_length=32, return_tensors="pt")
            txt_emb, _ = model.encode_text(text_enc["input_ids"].to(device), text_enc["attention_mask"].to(device))
            all_text_embs.append(txt_emb.cpu())

    all_image_embs = torch.cat(all_image_embs)
    all_text_embs = torch.cat(all_text_embs)

    # Text-to-image: each row is a query text, columns are images
    coarse_sim = all_text_embs @ all_image_embs.T  # (N, N)

    # ── Stage 2: Re-rank top-k image candidates with cross-attention ──
    print(f"Stage 2: Re-ranking top-{top_k} candidates...")
    score_matrix = torch.full((N, N), -1.0)

    with torch.no_grad():
        for i in tqdm(range(N), desc="Re-ranking (T→I)"):
            candidates = coarse_sim[i].topk(top_k).indices.tolist()

            text = all_texts[i]
            cand_images = [all_images[j] for j in candidates]
            text_enc = tokenizer([text] * top_k, padding="max_length", truncation=True, max_length=32, return_tensors="pt")
            image_enc = image_processor(images=cand_images, return_tensors="pt")

            itm_logits = model.compute_itm(
                model.encode_text(text_enc["input_ids"].to(device), text_enc["attention_mask"].to(device))[1],
                model.encode_image(image_enc["pixel_values"].to(device))[1],
            )
            itm_scores = torch.sigmoid(itm_logits).cpu()

            for rank, j in enumerate(candidates):
                score_matrix[i][j] = itm_scores[rank]

    def recall_at_k(score_mat, k):
        correct = 0
        for i in range(score_mat.size(0)):
            topk = score_mat[i].topk(k).indices
            if i in topk:
                correct += 1
        return correct / score_mat.size(0)

    print()
    print(f"=== [Text→Image] Stage 1 Only: Coarse Retrieval (N={N}) ===")
    print(f"  Recall@1:  {recall_at_k(coarse_sim, 1):.4f}")
    print(f"  Recall@5:  {recall_at_k(coarse_sim, 5):.4f}")
    print(f"  Recall@10: {recall_at_k(coarse_sim, 10):.4f}")

    print(f"\n=== [Text→Image] Two-Stage: Coarse + Re-Rank top-{top_k} (N={N}) ===")
    print(f"  Recall@1:  {recall_at_k(score_matrix, 1):.4f}")
    print(f"  Recall@5:  {recall_at_k(score_matrix, 5):.4f}")
    print(f"  Recall@10: {recall_at_k(score_matrix, 10):.4f}")

evaluate_text_to_image_1to1(model, test_data, num_samples=200, top_k=20)

Stage 1: Computing embeddings...
Stage 2: Re-ranking top-20 candidates...


Re-ranking (T→I): 100%|██████████| 200/200 [00:28<00:00,  7.10it/s]


=== [Text→Image] Stage 1 Only: Coarse Retrieval (N=200) ===
  Recall@1:  0.1750
  Recall@5:  0.8650
  Recall@10: 0.9350

=== [Text→Image] Two-Stage: Coarse + Re-Rank top-20 (N=200) ===
  Recall@1:  0.1650
  Recall@5:  0.8100
  Recall@10: 0.9300


# Evaluation: grouped

In [31]:
def eval_collate_grouped(batch):
    """Collate that also returns img_id for grouping."""
    texts = [x["caption"] for x in batch]
    images = [x["image"].convert("RGB") for x in batch]
    img_ids = [x["img_id"] for x in batch]
    return texts, images, img_ids

In [32]:
def evaluate_image_to_text_grouped(model, data, num_samples=200, top_k=20):
    """Two-stage retrieval with grouped recall (all captions for same image are correct)."""
    subset = data.select(range(min(num_samples, len(data))))
    loader = DataLoader(subset, batch_size=32, shuffle=False, collate_fn=eval_collate_grouped)

    all_texts = []
    all_images = []
    all_img_ids = []
    for texts, images, img_ids in loader:
        all_texts.extend(texts)
        all_images.extend(images)
        all_img_ids.extend(img_ids)

    # Build mapping: img_id -> list of indices
    group_to_indices = {}
    for idx, gid in enumerate(all_img_ids):
        if gid not in group_to_indices:
            group_to_indices[gid] = []
        group_to_indices[gid].append(idx)

    N = len(all_texts)

    # ── Stage 1: Encode all images and texts independently ──
    print("Stage 1: Computing embeddings...")
    all_image_embs = []
    all_text_embs = []
    model.eval()
    bs = 64
    with torch.no_grad():
        for start in range(0, N, bs):
            end = min(start + bs, N)
            image_enc = image_processor(images=all_images[start:end], return_tensors="pt")
            img_emb, _ = model.encode_image(image_enc["pixel_values"].to(device))
            all_image_embs.append(img_emb.cpu())

        for start in range(0, N, bs):
            end = min(start + bs, N)
            text_enc = tokenizer(all_texts[start:end], padding="max_length", truncation=True, max_length=32, return_tensors="pt")
            txt_emb, _ = model.encode_text(text_enc["input_ids"].to(device), text_enc["attention_mask"].to(device))
            all_text_embs.append(txt_emb.cpu())

    all_image_embs = torch.cat(all_image_embs)
    all_text_embs = torch.cat(all_text_embs)

    coarse_sim = all_image_embs @ all_text_embs.T

    # ── Stage 2: Re-rank top-k candidates with cross-attention ──
    print(f"Stage 2: Re-ranking top-{top_k} candidates...")
    score_matrix = torch.full((N, N), -1.0)

    with torch.no_grad():
        for i in tqdm(range(N), desc="Re-ranking"):
            candidates = coarse_sim[i].topk(top_k).indices.tolist()

            img = all_images[i]
            cand_texts = [all_texts[j] for j in candidates]
            image_enc = image_processor(images=[img] * top_k, return_tensors="pt")
            text_enc = tokenizer(cand_texts, padding="max_length", truncation=True, max_length=32, return_tensors="pt")

            itm_logits = model.compute_itm(
                model.encode_text(text_enc["input_ids"].to(device), text_enc["attention_mask"].to(device))[1],
                model.encode_image(image_enc["pixel_values"].to(device))[1],
            )
            itm_scores = torch.sigmoid(itm_logits).cpu()

            for rank, j in enumerate(candidates):
                score_matrix[i][j] = itm_scores[rank]

    def recall_at_k(score_mat, k, idx_to_group, group_to_idx):
        total_recall = 0.0
        for i in range(score_mat.size(0)):
            topk = set(score_mat[i].topk(k).indices.tolist())
            correct_set = set(group_to_idx[idx_to_group[i]])
            n_relevant = len(correct_set)
            n_retrieved = len(topk & correct_set)
            total_recall += n_retrieved / n_relevant
        return total_recall / score_mat.size(0)

    print()
    print(f"=== Stage 1 Only: Coarse Retrieval (N={N}, grouped) ===")
    print(f"  Recall@1:  {recall_at_k(coarse_sim, 1, all_img_ids, group_to_indices):.4f}")
    print(f"  Recall@5:  {recall_at_k(coarse_sim, 5, all_img_ids, group_to_indices):.4f}")
    print(f"  Recall@10: {recall_at_k(coarse_sim, 10, all_img_ids, group_to_indices):.4f}")

    print(f"\n=== Two-Stage: Coarse + Re-Rank top-{top_k} (N={N}, grouped) ===")
    print(f"  Recall@1:  {recall_at_k(score_matrix, 1, all_img_ids, group_to_indices):.4f}")
    print(f"  Recall@5:  {recall_at_k(score_matrix, 5, all_img_ids, group_to_indices):.4f}")
    print(f"  Recall@10: {recall_at_k(score_matrix, 10, all_img_ids, group_to_indices):.4f}")

evaluate_image_to_text_grouped(model, test_data, num_samples=200, top_k=20)

Stage 1: Computing embeddings...
Stage 2: Re-ranking top-20 candidates...


Re-ranking: 100%|██████████| 200/200 [00:27<00:00,  7.26it/s]


=== Stage 1 Only: Coarse Retrieval (N=200, grouped) ===
  Recall@1:  0.1850
  Recall@5:  0.8000
  Recall@10: 0.9600

=== Two-Stage: Coarse + Re-Rank top-20 (N=200, grouped) ===
  Recall@1:  0.1750
  Recall@5:  0.7800
  Recall@10: 0.9450


In [33]:
def evaluate_text_to_image_grouped(model, data, num_samples=200, top_k=20):
    """Two-stage TEXT-TO-IMAGE retrieval with grouped recall."""
    subset = data.select(range(min(num_samples, len(data))))
    loader = DataLoader(subset, batch_size=32, shuffle=False, collate_fn=eval_collate_grouped)

    all_texts = []
    all_images = []
    all_img_ids = []
    for texts, images, img_ids in loader:
        all_texts.extend(texts)
        all_images.extend(images)
        all_img_ids.extend(img_ids)

    # Build mapping: img_id -> list of indices
    group_to_indices = {}
    for idx, gid in enumerate(all_img_ids):
        if gid not in group_to_indices:
            group_to_indices[gid] = []
        group_to_indices[gid].append(idx)

    N = len(all_texts)

    # ── Stage 1: Encode all images and texts independently ──
    print("Stage 1: Computing embeddings...")
    all_image_embs = []
    all_text_embs = []
    model.eval()
    bs = 64
    with torch.no_grad():
        for start in range(0, N, bs):
            end = min(start + bs, N)
            image_enc = image_processor(images=all_images[start:end], return_tensors="pt")
            img_emb, _ = model.encode_image(image_enc["pixel_values"].to(device))
            all_image_embs.append(img_emb.cpu())

        for start in range(0, N, bs):
            end = min(start + bs, N)
            text_enc = tokenizer(all_texts[start:end], padding="max_length", truncation=True, max_length=32, return_tensors="pt")
            txt_emb, _ = model.encode_text(text_enc["input_ids"].to(device), text_enc["attention_mask"].to(device))
            all_text_embs.append(txt_emb.cpu())

    all_image_embs = torch.cat(all_image_embs)
    all_text_embs = torch.cat(all_text_embs)

    # Text-to-image coarse similarity: each row is a query text, columns are images
    coarse_sim = all_text_embs @ all_image_embs.T  # (N, N)

    # ── Stage 2: Re-rank top-k image candidates with cross-attention ──
    print(f"Stage 2: Re-ranking top-{top_k} candidates...")
    score_matrix = torch.full((N, N), -1.0)

    with torch.no_grad():
        for i in tqdm(range(N), desc="Re-ranking (T→I)"):
            # For text_i, get top-k image candidates
            candidates = coarse_sim[i].topk(top_k).indices.tolist()

            text = all_texts[i]
            cand_images = [all_images[j] for j in candidates]
            text_enc = tokenizer([text] * top_k, padding="max_length", truncation=True, max_length=32, return_tensors="pt")
            image_enc = image_processor(images=cand_images, return_tensors="pt")

            itm_logits = model.compute_itm(
                model.encode_text(text_enc["input_ids"].to(device), text_enc["attention_mask"].to(device))[1],
                model.encode_image(image_enc["pixel_values"].to(device))[1],
            )
            itm_scores = torch.sigmoid(itm_logits).cpu()

            for rank, j in enumerate(candidates):
                score_matrix[i][j] = itm_scores[rank]

    def recall_at_k(score_mat, k, idx_to_group, group_to_idx):
        total_recall = 0.0
        for i in range(score_mat.size(0)):
            topk = set(score_mat[i].topk(k).indices.tolist())
            correct_set = set(group_to_idx[idx_to_group[i]])
            n_relevant = len(correct_set)
            n_retrieved = len(topk & correct_set)
            total_recall += n_retrieved / n_relevant
        return total_recall / score_mat.size(0)

    print()
    print(f"=== [Text→Image] Stage 1 Only: Coarse Retrieval (N={N}, grouped) ===")
    print(f"  Recall@1:  {recall_at_k(coarse_sim, 1, all_img_ids, group_to_indices):.4f}")
    print(f"  Recall@5:  {recall_at_k(coarse_sim, 5, all_img_ids, group_to_indices):.4f}")
    print(f"  Recall@10: {recall_at_k(coarse_sim, 10, all_img_ids, group_to_indices):.4f}")

    print(f"\n=== [Text→Image] Two-Stage: Coarse + Re-Rank top-{top_k} (N={N}, grouped) ===")
    print(f"  Recall@1:  {recall_at_k(score_matrix, 1, all_img_ids, group_to_indices):.4f}")
    print(f"  Recall@5:  {recall_at_k(score_matrix, 5, all_img_ids, group_to_indices):.4f}")
    print(f"  Recall@10: {recall_at_k(score_matrix, 10, all_img_ids, group_to_indices):.4f}")

evaluate_text_to_image_grouped(model, test_data, num_samples=200, top_k=20)

Stage 1: Computing embeddings...
Stage 2: Re-ranking top-20 candidates...


Re-ranking (T→I): 100%|██████████| 200/200 [00:29<00:00,  6.79it/s]


=== [Text→Image] Stage 1 Only: Coarse Retrieval (N=200, grouped) ===
  Recall@1:  0.1730
  Recall@5:  0.8650
  Recall@10: 0.9350

=== [Text→Image] Two-Stage: Coarse + Re-Rank top-20 (N=200, grouped) ===
  Recall@1:  0.1620
  Recall@5:  0.8100
  Recall@10: 0.9300
